In [3]:
# -*- coding: utf-8 -*-
"""
predict.py
──────────
Script de prédiction YOLO sur de nouvelles images de sceaux byzantins.
- Charge le meilleur modèle (best.pt du fold choisi)
- Génère les bounding boxes sur les images
- Sauvegarde : images annotées + fichiers .txt YOLO
"""

!pip install ultralytics

from google.colab import drive
drive.mount('/content/drive')

import os
import cv2
import numpy as np
from ultralytics import YOLO
from pathlib import Path

# ─────────────────────────────────────────────
# CONFIGURATION  ← à adapter selon ton setup
# ─────────────────────────────────────────────
BASE_DIR        = '/content/drive/MyDrive/368651_Byzantine_monograms'
RUNS_DIR        = os.path.join(BASE_DIR, 'runs')

# Dossier contenant les nouvelles images à prédire
NEW_IMAGES_DIR = os.path.join(BASE_DIR, 'images', 'train')

# Dossier de sortie
OUTPUT_DIR      = os.path.join(BASE_DIR, 'predictions')
OUTPUT_IMAGES   = os.path.join(OUTPUT_DIR, 'images_annotees')
OUTPUT_LABELS   = os.path.join(OUTPUT_DIR, 'labels_yolo')

# Fold à utiliser (celui avec le meilleur mAP50 = fold 4 d'après tes résultats)
BEST_FOLD       = 4

# Paramètres de prédiction
IMG_SIZE        = 640
CONF_THRESHOLD  = 0.25   # seuil de confiance (0.25 = défaut YOLO)
IOU_THRESHOLD   = 0.45   # seuil NMS
DEVICE          = 0      # GPU


# ─────────────────────────────────────────────
# FONCTIONS
# ─────────────────────────────────────────────

def get_model(fold_idx):
    """Charge le meilleur modèle du fold choisi."""
    weights = os.path.join(RUNS_DIR, f'fold_{fold_idx}', 'weights', 'best.pt')
    if not os.path.exists(weights):
        raise FileNotFoundError(f"Poids introuvables : {weights}")
    print(f"  ✓ Modèle chargé : {weights}")
    return YOLO(weights)


def get_image_paths(images_dir):
    """Récupère tous les chemins d'images valides dans le dossier."""
    extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}
    paths = [
        str(p) for p in Path(images_dir).rglob('*')
        if p.suffix.lower() in extensions
        and not p.name.startswith('._')  # exclut les fichiers cachés macOS
    ]
    if not paths:
        raise ValueError(f"Aucune image trouvée dans : {images_dir}")
    print(f"  ✓ {len(paths)} image(s) trouvée(s)")
    return sorted(paths)


def save_yolo_labels(result, output_labels_dir, img_name):
    """
    Sauvegarde les prédictions au format YOLO :
    <class_id> <x_center> <y_center> <width> <height> <confidence>
    """
    txt_path = os.path.join(output_labels_dir, img_name + '.txt')
    boxes = result.boxes

    if boxes is None or len(boxes) == 0:
        # Fichier vide si aucune détection
        open(txt_path, 'w').close()
        return 0

    with open(txt_path, 'w') as f:
        for box in boxes:
            cls_id = int(box.cls.item())
            conf   = float(box.conf.item())
            # xywhn = coordonnées normalisées [0,1]
            x_c, y_c, w, h = box.xywhn[0].tolist()
            f.write(f"{cls_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f} {conf:.4f}\n")

    return len(boxes)


def save_annotated_image(result, output_images_dir, img_name):
    """Sauvegarde l'image avec les bounding boxes dessinées."""
    annotated = result.plot()  # image BGR avec boxes + labels + scores
    out_path  = os.path.join(output_images_dir, img_name + '_pred.jpg')
    cv2.imwrite(out_path, annotated)
    return out_path


def predict_all(model, image_paths):
    """Lance la prédiction sur toutes les images et sauvegarde les résultats."""
    os.makedirs(OUTPUT_IMAGES, exist_ok=True)
    os.makedirs(OUTPUT_LABELS, exist_ok=True)

    total_detections = 0
    summary = []

    for img_path in image_paths:
        img_name = Path(img_path).stem
        print(f"\n  → Traitement : {Path(img_path).name}")

        results = model.predict(
            source    = img_path,
            imgsz     = IMG_SIZE,
            conf      = CONF_THRESHOLD,
            iou       = IOU_THRESHOLD,
            device    = DEVICE,
            verbose   = False,
        )

        result = results[0]  # une image = un résultat
        n_det  = len(result.boxes) if result.boxes is not None else 0

        # Sauvegarde labels .txt
        save_yolo_labels(result, OUTPUT_LABELS, img_name)

        # Sauvegarde image annotée
        out_img = save_annotated_image(result, OUTPUT_IMAGES, img_name)

        print(f"     {n_det} détection(s)  →  {out_img}")
        total_detections += n_det
        summary.append({'image': Path(img_path).name, 'detections': n_det})

    return summary, total_detections


def print_summary(summary, total):
    """Affiche un résumé des prédictions."""
    print(f"\n{'='*60}")
    print(f"  RÉSUMÉ DES PRÉDICTIONS")
    print(f"{'='*60}")
    print(f"  Images traitées  : {len(summary)}")
    print(f"  Détections total : {total}")
    print(f"  Moyenne/image    : {total / len(summary):.1f}")
    print(f"\n  Détail par image :")
    for s in summary:
        print(f"    {s['image']:40s}  →  {s['detections']} détection(s)")
    print(f"\n  Images annotées  : {OUTPUT_IMAGES}")
    print(f"  Labels YOLO      : {OUTPUT_LABELS}")


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def main():
    print(f"\n{'='*60}")
    print(f"  PRÉDICTION BYZANTINE SEAL DETECTOR")
    print(f"{'='*60}\n")

    print(f"[1/3] Chargement du modèle (fold {BEST_FOLD})...")
    model = get_model(BEST_FOLD)

    print(f"\n[2/3] Chargement des images depuis :\n  {NEW_IMAGES_DIR}")
    image_paths = get_image_paths(NEW_IMAGES_DIR)

    print(f"\n[3/3] Prédiction en cours...")
    summary, total = predict_all(model, image_paths)

    print_summary(summary, total)
    print("\n✓ Prédictions terminées !")


if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

  PRÉDICTION BYZANTINE SEAL DETECTOR

[1/3] Chargement du modèle (fold 4)...
  ✓ Modèle chargé : /content/drive/MyDrive/368651_Byzantine_monograms/runs/fold_4/weights/best.pt

[2/3] Chargement des images depuis :
  /content/drive/MyDrive/368651_Byzantine_monograms/images/train
  ✓ 351 image(s) trouvée(s)

[3/3] Prédiction en cours...

  → Traitement : 1112503_2026-01-28_001.jpg
     4 détection(s)  →  /content/drive/MyDrive/368651_Byzantine_monograms/predictions/images_annotees/1112503_2026-01-28_001_pred.jpg

  → Traitement : 1112503_2026-01-28_002.jpg
     9 détection(s)  →  /content/drive/MyDrive/368651_Byzantine_monograms/predictions/images_annotees/1112503_2026-01-28_002_pred.jpg

  → Traitement : 1112503_2026-01-28_003.png
     7 détection(s)  →  /content/drive/MyDrive/368651_Byzantine_monograms/predictions/images_annotees/1112503_2026-01-28_003_pred.j